# 05 - Run the merged pipeline

Notebook 02 now contains both Global ReID and Zones. Set RUN_PIPELINE to True only when you intentionally want to run the full video pipeline.


In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebook':
    PROJECT_ROOT = PROJECT_ROOT.parent
APP_PATH = PROJECT_ROOT / 'Output' / 'app' / 'streamlit_app.py'

RUN_PIPELINE = True  # Change to True only for an intentional full run.
START_STREAMLIT = False
PIPELINE = [
    '00_Project_Setup.ipynb',
    '01_Local_Detection_Tracking.ipynb',
    '02_Global_Fusion_and_Zones.ipynb',
    '03_Retail_Analytics_Agent.ipynb',
    '04_Streamlit_Dashboard.ipynb',
]
print('Pipeline:', ' -> '.join(PIPELINE))


In [ ]:
if RUN_PIPELINE:
    import nbformat
    from nbclient import NotebookClient
    from nbclient.exceptions import CellExecutionError

    for notebook_name in PIPELINE:
        notebook_path = PROJECT_ROOT / 'Notebook' / notebook_name
        print(f'Running {notebook_name} ...')
        notebook = nbformat.read(notebook_path, as_version=4)
        client = NotebookClient(
            notebook, timeout=None, kernel_name='python3',
            resources={'metadata': {'path': str(PROJECT_ROOT)}}
        )
        try:
            client.execute()
        except CellExecutionError as error:
            if notebook_name == '02_Global_Fusion_and_Zones.ipynb' and 'ReID stopped:' in str(error):
                nbformat.write(notebook, notebook_path)
                print('ReID is safely blocked; camera-local zones were saved, so continuing with analytics and dashboard.')
                continue
            raise
        nbformat.write(notebook, notebook_path)
    print('Pipeline completed.')
else:
    print('No video or model code was started. Set RUN_PIPELINE = True when ready.')


In [ ]:
if START_STREAMLIT:
    command = [sys.executable, '-m', 'streamlit', 'run', str(APP_PATH), '--server.port', '8501']
    process = subprocess.Popen(command, cwd=PROJECT_ROOT)
    print(f'Streamlit started with PID {process.pid}. Open http://localhost:8501')
